# Installing Libraries

In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'


In [2]:
!pip install bs4
!pip install wikitextparser
!pip install sentence-transformers
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [3]:
import requests
from bs4 import BeautifulSoup
import wikitextparser as wtp
from transformers import pipeline
import time
import json
import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize
import faiss
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

# Scraping nomadicMatt.com

In [4]:

HEADERS = {"User-Agent":"Mozilla/5.0"}
BASE = "https://www.nomadicmatt.com"


In [5]:
guide_urls = [
    "https://www.nomadicmatt.com/travel-guides/europe-travel-tips/",
    "https://www.nomadicmatt.com/travel-guides/southeast-asia-travel-tips/",
    "https://www.nomadicmatt.com/travel-guides/",
    "https://www.nomadicmatt.com/travel-guides/caribbean-travel-tips/",
    "https://www.nomadicmatt.com/travel-guides/central-america-travel-tips/"
]

link_cache_path = "guide_article_links.json"

if os.path.exists(link_cache_path):
    with open(link_cache_path, "r", encoding="utf-8") as f:
        guide_to_links = json.load(f)
else:
    guide_to_links = {}

def is_valid_article_url(href):
    return (
        href and
        href.startswith("https://www.nomadicmatt.com/travel-guides/") and
        href.count("/") > 4 and
        not any(x in href for x in ["#", "mailto:", ".pdf"])
    )

for guide_url in guide_urls:
    if guide_url in guide_to_links:
        print(f"[CACHE] Skipping already scanned: {guide_url}")
        continue

    print(f"[FETCH] Scraping guide: {guide_url}")
    try:
        res = requests.get(guide_url, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")
        links = set()

        for a in soup.find_all("a", href=True):
            href = a['href'].strip()
            if is_valid_article_url(href):
                links.add(href)

        guide_to_links[guide_url] = sorted(links)

    except Exception as e:
        print(f"Error scraping {guide_url}: {e}")

with open(link_cache_path, "w", encoding="utf-8") as f:
    json.dump(guide_to_links, f, indent=2)

print(f"Finished. Saved article links in: {link_cache_path}")


[FETCH] Scraping guide: https://www.nomadicmatt.com/travel-guides/europe-travel-tips/
[FETCH] Scraping guide: https://www.nomadicmatt.com/travel-guides/southeast-asia-travel-tips/
[FETCH] Scraping guide: https://www.nomadicmatt.com/travel-guides/
[FETCH] Scraping guide: https://www.nomadicmatt.com/travel-guides/caribbean-travel-tips/
[FETCH] Scraping guide: https://www.nomadicmatt.com/travel-guides/central-america-travel-tips/
Finished. Saved article links in: guide_article_links.json


In [6]:
link_cache_path = "guide_article_links.json"
if not os.path.exists(link_cache_path):
    raise FileNotFoundError(f"{link_cache_path} not found.")
with open(link_cache_path, "r", encoding="utf-8") as f:
    guide_to_links = json.load(f)

all_article_urls = [url for urls in guide_to_links.values() for url in urls]
print(f"Total article links found: {len(all_article_urls)}")

Total article links found: 96


In [7]:
output_dir = "normadicMatt_articles"
os.makedirs(output_dir, exist_ok=True)

def clean_filename(text):
    return re.sub(r'[\\/*?:"<>|]', "_", text).strip().replace(" ", "_")[:100]

def show_article_structure(url, index):
    print(f"\nScraping: {url}")
    try:
        res = requests.get(url, timeout=10)
        res.raise_for_status()
        if "nomadicmatt.com" not in res.url:
            print(f"Redirected to non-target domain: {res.url}")
            return
    except Exception as e:
        print(f"Failed to fetch {url}: {e}")
        return

    soup = BeautifulSoup(res.text, "html.parser")

    title_tag = soup.find("h1")
    if not title_tag:
        print(f"No title found — skipping: {url}")
        return

    title = title_tag.get_text(strip=True).strip()
    if not title:
        print(f"No title found — skipping: {url}")
        return

    filename = clean_filename(title)

    content_div = soup.find("div", class_="entry-content") or soup.find("div", class_="kt-inside-inner-col")
    if not content_div:
        print(f"No content found — skipping: {url}")
        return

    sections = {"Introduction": []}
    current = "Introduction"

    for tag in content_div.find_all(["h2", "h3", "h4", "h5", "p", "ul"]):
        if tag.name in ["h2", "h3", "h4", "h5"]:
            current = tag.get_text(strip=True) or current
            sections.setdefault(current, [])
        elif tag.name == "p":
            text = tag.get_text(strip=True)
            if text:
                sections[current].append(text)
        elif tag.name == "ul" and "wp-block-list" in (tag.get("class") or []):
            links, items = [], []
            for li in tag.find_all("li"):
                a = li.find("a", href=True)
                if a:
                    links.append({"text": a.get_text(strip=True), "href": a["href"]})
                else:
                    t = li.get_text(strip=True)
                    if t:
                        items.append(t)
            block = {}
            if links:
                block["links"] = links
            if items:
                block["items"] = items
            if block:
                sections[current].append(block)

    article_data = {
        "title": title,
        "sections": sections
    }

    if not any(sections.values()):
        print("No meaningful sections parsed — skipping file save.")
        return

    file_path = os.path.join(output_dir, f"{filename}.json")
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(article_data, f, indent=2, ensure_ascii=False)
    print(f"Saved: {file_path}")
    time.sleep(1)


In [8]:
for idx, url in enumerate(all_article_urls, start=1):
    print(f"\n======= [{idx}] =======")
    show_article_structure(url, idx)
    time.sleep(1)



======= [1] =======

Scraping: https://www.nomadicmatt.com/travel-guides/albania-travel-guide/
Saved: normadicMatt_articles/Albania_Travel_Guide.json

======= [2] =======

Scraping: https://www.nomadicmatt.com/travel-guides/austria-travel-guide/
Saved: normadicMatt_articles/Austria_Travel_Guide.json

======= [3] =======

Scraping: https://www.nomadicmatt.com/travel-guides/belarus-travel-guide/
Saved: normadicMatt_articles/Belarus_Travel_Guide.json

======= [4] =======

Scraping: https://www.nomadicmatt.com/travel-guides/belgium/
Saved: normadicMatt_articles/Belgium_Travel_Guide.json

======= [5] =======

Scraping: https://www.nomadicmatt.com/travel-guides/bosnia-herzegovina-travel-guide/
Saved: normadicMatt_articles/Bosnia_&_Herzegovina_Travel_Guide.json

======= [6] =======

Scraping: https://www.nomadicmatt.com/travel-guides/bulgaria/
Saved: normadicMatt_articles/Bulgaria_Travel_Guide.json

======= [7] =======

Scraping: https://www.nomadicmatt.com/travel-guides/croatia-travel-guide

In [9]:
with open("normadicMatt_articles/Mexico_Travel_Guide.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False))


{
  "title": "Mexico Travel Guide",
  "sections": {
    "Introduction": [
      "While most people visit Mexico for its big tourist centers likeTulum, Cabo,Cancun, or Cozumel, there’s a lot more to the country than just its luxurious resorts.",
      "I was late to exploring Mexico but, when I did, I fell in love with it. Mexico is an incredible destination with a rich history, amazing food, and friendly people.",
      "It’s an awesome country to backpack around, drive through, or just vacation in. There’s a ton of stuff to do here, and the locals are some of the friendliest people on the planet.",
      "From Mayan ruins to pristine beaches toMexico City’sart and food andOaxaca’smezcal scene, Mexico has it all.",
      "And the food? World-class. Gorge yourself on delicious tacos, tostadas, tamales, sopas, seafood, and mole (to name a few items from Mexico’s very long list of traditional dishes).",
      "I could go on forever as to why I love this country. Whatever amount of time yo

# Scraping Guide to Pakistan

In [10]:
main_url = "https://guidetopakistan.pk/"

response = requests.get(main_url)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")

    extracted_links = []

    for item in soup.select(".elementor-image-box-title a"):
        link = item['href']
        extracted_links.append(link)

    print("Extracted Links:")
    for link in extracted_links:
        print(link)
else:
    print(f"Failed to retrieve the page. Status code: {response.status_code}")


Extracted Links:
https://guidetopakistan.pk/destination/lahore/
https://guidetopakistan.pk/destination/karachi-travel-guide/
https://guidetopakistan.pk/destination/taxila/
https://guidetopakistan.pk/destination/arang-kel/
https://guidetopakistan.pk/destination/sehwan-sharif/
https://guidetopakistan.pk/destination/larkana/
https://guidetopakistan.pk/tour/sharan-honeymoon-tour-package/
https://guidetopakistan.pk/tour/chitral-honeymoon-tour-package/
https://guidetopakistan.pk/tour/naran-honeymoon-tour-package/
https://guidetopakistan.pk/tour/kashmir-honeymoon-tour-package/
https://guidetopakistan.pk/tour/murree-honeymoon-tour-package/
https://guidetopakistan.pk/tour/hunza-skardu-tour/
https://guidetopakistan.pk/tour/golf-tour/


In [11]:
output_directory = "guide_to_pakistan_articles"
os.makedirs(output_directory, exist_ok=True)

def extract_paragraphs(content_div):
    paragraphs = []
    for elem in content_div.descendants:
        if elem.name in ["p", "b", "h4"] or (elem.name is None and str(elem).strip()):
            for tag in getattr(elem, 'find_all', lambda *_: [])(["a", "span"]):
                tag.unwrap()
            text = elem.get_text(strip=True) if hasattr(elem, "get_text") else str(elem).strip()
            if text:
                paragraphs.append(text)
    return paragraphs

In [12]:
def scrape_tour_pages(tour_urls):

    for url in tour_urls:
        try:
            res = requests.get(url)
            res.raise_for_status()
        except Exception as e:
            print(f"Failed to fetch {url}: {e}")
            continue

        page = BeautifulSoup(res.text, "html.parser")

        title_tag = page.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else "tour_page"


        filename =clean_filename(title)
        file_path = os.path.join(output_directory, f"{filename}.json")

        tabs = []
        for tab_title in page.select("div.elementor-tab-title"):
            title_id = tab_title.get("id")
            title_tag = tab_title.select_one("a.elementor-toggle-title")
            if not title_id or not title_tag:
                continue

            heading = title_tag.get_text(strip=True)
            content_id = title_id.replace("title", "content")
            content_div = page.find("div", id=content_id)

            if content_div:
                paragraphs = extract_paragraphs(content_div)
                if paragraphs:
                    tabs.append({"heading": heading, "paragraphs": paragraphs})

        tour_data = {"title": title, "tabs": tabs}

        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(tour_data, f, indent=2, ensure_ascii=False)
        print(f"Saved tour: {file_path}")
        time.sleep(1)


In [13]:
def extract_content(page, banned_phrases):
    content = []
    for tag in page.find_all(["h1", "h2", "h3", "p"]):
        text = tag.get_text(strip=True)
        if text and text not in banned_phrases:
            if tag.name in ["h1", "h2", "h3"]:
                content.append(f"\n{text}\n")
            else:
                content.append(text)
    return content

In [14]:
def extract_sections(page):
    sections = []
    for heading_tag in page.find_all(["h2", "h3"]):
        heading = heading_tag.get_text(strip=True)
        paragraph = ""

        next_p = heading_tag.find_next_sibling()
        while next_p and next_p.name != "p":
            next_p = next_p.find_next_sibling()
        if next_p and next_p.name == "p":
            paragraph = next_p.get_text(strip=True)

        if heading:
            sections.append({"heading": heading})
        if paragraph:
            sections.append({"paragraph": paragraph})
    return sections

In [15]:
def scrape_destination_pages(main_url):
    response = requests.get(main_url)
    if response.status_code != 200:
        print("Failed to load main page.")
        return

    soup = BeautifulSoup(response.text, "html.parser")
    destination_links = [item['href'] for item in soup.select(".elementor-image-box-title a") if "destination" in item['href']]

    banned_phrases = {
        "Customised Tour Package", "Corporate Tour", "HoneyMoon Package",
        "Pakistan Cultural and Religious tour", "Gilgit Baltistan",
        "Khyber Pakhtunkhwa", "Islamabad", "Punjab", "Sindh",
        "Balochistan", "Kashmir", "@Copyright 2025 Guide To Pakistan | Powered by CyberX Studio",
        "WhatsApp us", "Recent posts", "Get up to 30% off on all customized Tour packages."
    }

    for url in destination_links:

        try:
            res = requests.get(url)
            res.raise_for_status()
            page = BeautifulSoup(res.text, "html.parser")
        except Exception as e:
            print(f"Failed to fetch {url}: {e}")
            continue

        title_tag = page.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else url.split("/")[-2]


        content = extract_content(page, banned_phrases)
        sections = extract_sections(page)

        filename =clean_filename(title)
        file_path = os.path.join("guide_to_pakistan_articles", f"{filename}.json")

        with open(file_path, "w", encoding="utf-8") as f:
            json.dump({
                "title": title,
                "content": content,
                "sections": sections
            }, f, indent=2, ensure_ascii=False)

        print(f"Saved destination: {file_path}")
        time.sleep(1)


In [16]:
if __name__ == "__main__":
    main_url = "https://guidetopakistan.pk/"
    tour_urls = [
    "https://guidetopakistan.pk/tour/sharan-honeymoon-tour-package/",
    "https://guidetopakistan.pk/tour/chitral-honeymoon-tour-package/",
    "https://guidetopakistan.pk/tour/naran-honeymoon-tour-package/",
    "https://guidetopakistan.pk/tour/kashmir-honeymoon-tour-package/",
    "https://guidetopakistan.pk/tour/murree-honeymoon-tour-package/",
    "https://guidetopakistan.pk/tour/hunza-skardu-tour/",
    "https://guidetopakistan.pk/tour/golf-tour/"
]
    tour_data = scrape_tour_pages(tour_urls)
    destination_data = scrape_destination_pages(main_url)

    print("\n All pages scraped and saved to folder guide_to_pakistan_articles.")


Saved tour: guide_to_pakistan_articles/Sharan_Honeymoon_Tour_Package.json
Saved tour: guide_to_pakistan_articles/Chitral_Honeymoon_Tour_Package.json
Saved tour: guide_to_pakistan_articles/Naran_Honeymoon_Tour_Package.json
Saved tour: guide_to_pakistan_articles/Kashmir_Honeymoon_Tour_Package.json
Saved tour: guide_to_pakistan_articles/Murree_Honeymoon_Tour_Package.json
Saved tour: guide_to_pakistan_articles/Hunza_Skardu_Tour.json
Saved tour: guide_to_pakistan_articles/Golf_Tour.json
Saved destination: guide_to_pakistan_articles/Lahore.json
Saved destination: guide_to_pakistan_articles/Karachi.json
Saved destination: guide_to_pakistan_articles/Taxila.json
Saved destination: guide_to_pakistan_articles/Arang_Kel.json
Saved destination: guide_to_pakistan_articles/Sehwan_Sharif.json
Saved destination: guide_to_pakistan_articles/Larkana.json

 All pages scraped and saved to folder guide_to_pakistan_articles.


In [17]:
with open("guide_to_pakistan_articles/Sharan_Honeymoon_Tour_Package.json", "r", encoding="utf-8") as f:
    answer = json.load(f)

print(json.dumps(answer, indent=2, ensure_ascii=False))


{
  "title": "Sharan Honeymoon Tour Package",
  "tabs": [
    {
      "heading": "Overview",
      "paragraphs": [
        "Enjoy the serene beauty of Sharan with your spouse. Offering a perfect blend of nature, adventure, and luxury – the Sharan honeymoon tour package promises a beautiful experience. Nestled in the tranquil hills of Sharan Forest Resort, this honeymoon tour is ideal for couples seeking an intimate escape, offering scenic views and exciting treks in the heart of nature. Whether you opt for customized honeymoon tour packages for Sharan or explore the beauty of",
        "Punjab",
        "’s hidden gems, the Sharan honeymoon tours are tailored for couples who cherish peaceful moments and adventurous pursuits."
      ]
    },
    {
      "heading": "Location",
      "paragraphs": [
        "Located in the scenic Kaghan Valley, Sharan is a beautiful mountainous area famous for its lush green forests and captivating landscapes. Known for its cool weather and natural beauty

# Scraping wikivoyage

In [18]:
output_dir = "wikivoyage_articles"
os.makedirs(output_dir, exist_ok=True)

API_URL = "https://en.wikivoyage.org/w/api.php"
HEADERS = {
    "User-Agent": "RAGDataCollector/1.0 (contact@example.com)" # custom user-agent string
}

def get_wikitext(title, retries=3, delay=3):
    for attempt in range(retries):
        try:
            resp = requests.get(API_URL, headers=HEADERS, params={
                "action": "query",
                "format": "json",
                "titles": title,
                "prop": "revisions",
                "rvprop": "content",
                "rvslots": "*",
                "formatversion": "2"
            })
            resp.raise_for_status()
            pages = resp.json()["query"]["pages"]
            return pages[0]["revisions"][0]["slots"]["main"]["content"]
        except requests.exceptions.HTTPError as e:
            if resp.status_code == 429:
                print("Rate limited. Retrying after delay...")
                time.sleep(delay)
            else:
                print(f"HTTP error for {title}: {e}")
                return None
        except Exception as e:
            print(f"Error fetching {title}: {e}")
            return None
    return None

def parse_sections(wikitext):
    doc = wtp.parse(wikitext)
    sections = {}
    for sect in doc.get_sections(include_subsections=True):
        title = (sect.title or "").strip("=").strip() or "Introduction"
        sec_data = {"text": sect.plain_text().strip()}

        lists = [lst.items for lst in sect.get_lists()]
        if lists:
            sec_data["lists"] = lists

        tables = [tbl.data() for tbl in sect.tables]
        if tables:
            sec_data["tables"] = tables

        sections[title] = sec_data
    return sections

def scrape_countries(countries):
    for country in countries:
        print(f"\nScraping: {country}")
        wikitext = get_wikitext(country)
        if not wikitext:
            print(f"Skipping {country} due to failed fetch.")
            continue

        data = parse_sections(wikitext)
        filename = country.lower().replace(" ", "_") + ".json"
        filepath = os.path.join(output_dir, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        print(f"Saved to: {filepath}")
        time.sleep(2)

if __name__ == "__main__":
    countries = [
        "Pakistan", "Turkey", "England", "Japan", "Australia",
        "United States of America", "Germany", "France", "India",
        "South Africa", "Saudi Arabia", "Iran", "China"
    ]
    scrape_countries(countries)



Scraping: Pakistan
Saved to: wikivoyage_articles/pakistan.json

Scraping: Turkey
Saved to: wikivoyage_articles/turkey.json

Scraping: England
Saved to: wikivoyage_articles/england.json

Scraping: Japan
Saved to: wikivoyage_articles/japan.json

Scraping: Australia
Saved to: wikivoyage_articles/australia.json

Scraping: United States of America
Saved to: wikivoyage_articles/united_states_of_america.json

Scraping: Germany
Saved to: wikivoyage_articles/germany.json

Scraping: France
Saved to: wikivoyage_articles/france.json

Scraping: India
Saved to: wikivoyage_articles/india.json

Scraping: South Africa
Saved to: wikivoyage_articles/south_africa.json

Scraping: Saudi Arabia
Saved to: wikivoyage_articles/saudi_arabia.json

Scraping: Iran
Saved to: wikivoyage_articles/iran.json

Scraping: China
Saved to: wikivoyage_articles/china.json


In [19]:
with open("wikivoyage_articles/united_states_of_america.json", "r", encoding="utf-8") as f:
    wiki = json.load(f)

print(json.dumps(wiki, indent=2, ensure_ascii=False))


{
  "Introduction": {
    "text": "The United States of America spans a continent and numerous islands: its diverse geography comprises vast uninhabited areas of natural beauty punctuated by cities ringed by sprawling suburbs. Its wide array of tourist destinations includes the skyscrapers of Manhattan and Chicago, the natural wonders of Yellowstone and Alaska, the canyonlands of the Southwest, the towering volcanoes and rugged coastlines of the Pacific Northwest, and the warm, sunny beaches of Florida, Hawaii, and Southern California.\n\nRegarded as the world's most powerful and influential country, the U.S. plays a dominant role in the world's cultural landscape. Its landmarks and landscapes feature in countless books, movies and television programs viewed around the world. With a history of mass immigration dating from the 17th century, the U.S. is famous for being a \"melting pot\" of cultures from around the world."
  },
  "Regions": {
    "text": "==Regions==\nWikivoyage organize

# RAG

## Preprocessing data of files

In [20]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("omw-1.4")

stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [37]:
lemmatizer = WordNetLemmatizer()

def clean_html(raw_html):
    soup = BeautifulSoup(raw_html, "html.parser")
    return soup.get_text(separator=" ", strip=True)


def remove_duplicate_sentences(text):
    sentences = sent_tokenize(text)
    seen = set()
    unique_sentences = []

    for s in sentences:
        s_clean = s.strip()
        s_key = s_clean.lower()
        if s_clean and s_key not in seen:
            seen.add(s_key)
            unique_sentences.append(s_clean)

    return " ".join(unique_sentences)

def preprocess_text_for_chunking(text):
    text = clean_html(text)
    text = text.replace("’", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    text = re.sub(r'([.,;:!?])([^\s])', r'\1 \2', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return remove_duplicate_sentences(text)

In [38]:
def preprocess_text_for_query_expansion(text):
    text = clean_html(text)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))

    tokens = nltk.word_tokenize(text)
    clean_tokens = []
    prev_token = None

    for token in tokens:
        if token in stop_words:
            continue
        lemma = lemmatizer.lemmatize(token)
        if lemma != prev_token:
            clean_tokens.append(lemma)
            prev_token = lemma

    return " ".join(clean_tokens)


In [39]:
def extract_text(data):
    text_chunks = []

    if "sections" in data and isinstance(data["sections"], dict):
        for section in data["sections"].values():
            if isinstance(section, list):
                for item in section:
                    if isinstance(item, str):
                        text_chunks.append(item)
                    elif isinstance(item, dict):
                        if "text" in item and isinstance(item["text"], str):
                            text_chunks.append(item["text"])
                        if "items" in item and isinstance(item["items"], list):
                            text_chunks.extend(
                                i for i in item["items"] if isinstance(i, str)
                            )
            elif isinstance(section, str):
                text_chunks.append(section)

    elif isinstance(data, dict):
        for key, value in data.items():
            if isinstance(value, dict):
                if "text" in value and isinstance(value["text"], str):
                    text_chunks.append(value["text"])
                if "items" in value and isinstance(value["items"], list):
                    text_chunks.extend([i for i in value["items"] if isinstance(i, str)])
                if "paragraphs" in value and isinstance(value["paragraphs"], list):
                    text_chunks.extend([p for p in value["paragraphs"] if isinstance(p, str)])
            elif isinstance(value, list):
                for item in value:
                    if isinstance(item, str):
                        text_chunks.append(item)
                    elif isinstance(item, dict):
                        text_chunks.extend(extract_text(item).split('\n'))
            elif isinstance(value, str):
                text_chunks.append(value)

    elif isinstance(data, list):
        for item in data:
            if isinstance(item, str):
                text_chunks.append(item)
            elif isinstance(item, dict):
                text_chunks.extend(extract_text(item).split('\n'))

    return "\n".join(text_chunks)


In [40]:

def preprocess_all_articles(base_dirs):
    all_docs = []
    total_files = 0
    skipped_files = []

    for folder in base_dirs:
        print(f"\nProcessing folder: {folder}")
        for filename in os.listdir(folder):
            if not filename.lower().endswith(".json"):
                continue

            total_files += 1
            file_path = os.path.join(folder, filename)

            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                title = data.get("title", filename.replace(".json", ""))
                raw_text = extract_text(data)
                cleaned_text = preprocess_text_for_chunking(raw_text)

                if len(cleaned_text.split()) > 50:
                    all_docs.append({
                        "title": title.strip(),
                        "text": cleaned_text
                    })
                else:
                    skipped_files.append((filename, "Too short after cleaning"))

            except json.JSONDecodeError:
                skipped_files.append((filename, "Invalid JSON"))
            except Exception as e:
                skipped_files.append((filename, f"Other error: {str(e)}"))

    print(f"\nPreprocessed: {len(all_docs)} / {total_files} files")
    if skipped_files:
        print("\nSkipped Files:")
        for name, reason in skipped_files:
            print(f" - {name}: {reason}")

    return all_docs

In [41]:
base_dirs = [
    "/content/guide_to_pakistan_articles",
    "/content/normadicMatt_articles",
    "/content/wikivoyage_articles"
]

docs = preprocess_all_articles(base_dirs)


Processing folder: /content/guide_to_pakistan_articles

Processing folder: /content/normadicMatt_articles

Processing folder: /content/wikivoyage_articles

Preprocessed: 121 / 121 files


In [42]:
output_path = "/content/cleaned_docs.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for doc in docs:
        json.dump(doc, f, ensure_ascii=False)
        f.write("\n")

print(f"Saved {len(docs)} documents to {output_path}")


Saved 121 documents to /content/cleaned_docs.jsonl


# Creating Chunks of Data

In [43]:
from nltk.tokenize import sent_tokenize

def split_into_chunks(text, chunk_size=400, overlap=100, max_words=500):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_len = 0

    for sentence in sentences:
        words = sentence.split()
        sent_len = len(words)

        if current_len + sent_len > chunk_size:
            if current_chunk:
                chunk_text = " ".join(current_chunk)
                if len(chunk_text.split()) <= max_words:
                    chunks.append(chunk_text)
                else:
                    chunks.append(" ".join(chunk_text.split()[:max_words]))

            overlap_sentences = []
            total_overlap = 0
            for prev_sentence in reversed(current_chunk):
                w_len = len(prev_sentence.split())
                if total_overlap + w_len <= overlap:
                    overlap_sentences.insert(0, prev_sentence)
                    total_overlap += w_len
                else:
                    break

            current_chunk = overlap_sentences + [sentence]
            current_len = total_overlap + sent_len
        else:
            current_chunk.append(sentence)
            current_len += sent_len

    if current_chunk:
        chunk_text = " ".join(current_chunk)
        if len(chunk_text.split()) <= max_words:
            chunks.append(chunk_text)
        else:
            chunks.append(" ".join(chunk_text.split()[:max_words]))

    return chunks


In [44]:
def prepare_chunked_docs(all_docs, chunk_size=400, overlap=100, max_words=500):
    chunked = []
    for doc in all_docs:
        title = doc.get('title', 'unknown')
        text = doc.get('text', '')
        chunks = split_into_chunks(text, chunk_size, overlap, max_words)

        for idx, chunk in enumerate(chunks):
            chunked.append({
                "title": title,
                "chunk_id": f"{title.replace(' ', '_').lower()}_{idx}",
                "text": chunk
            })

    return chunked


# Saving chunking data in file

In [45]:
def save_chunked_data(chunked_docs, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for entry in chunked_docs:
            json.dump(entry, f)
            f.write("\n")


In [46]:
if __name__ == "__main__":

    chunked_docs = prepare_chunked_docs(docs, chunk_size=400, overlap=100)

    save_chunked_data(chunked_docs, "chunked_documents.jsonl")

    print(f"\nChunked documents saved to 'chunked_documents.jsonl'")



Chunked documents saved to 'chunked_documents.jsonl'


## Embedding, Indexing, Retreiving

In [47]:
def embed_texts(texts, model):
    emb = model.encode(texts, show_progress_bar=True)
    faiss.normalize_L2(emb)
    return emb

def build_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

def retrieval_top_k(query_text, model, index, documents, k=4):
    query_emb = model.encode([query_text], normalize_embeddings=True)
    scores, indices = index.search(query_emb, k)
    return indices[0], scores[0]



## Summarizing

In [66]:
from transformers import BartTokenizer, BartForConditionalGeneration

bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

def summarize_text(text, model=bart_model, tokenizer=bart_tokenizer, max_input_length=1024, max_output_length=250):
    input_text = text.strip().replace("\n", " ")
    inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=max_input_length, truncation=True)

    summary_ids = model.generate(inputs, max_length=max_output_length, num_beams=4, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary.strip()


In [62]:
def compute_term_distribution(docs):
    tokens = []
    for doc in docs:
        cleaned = preprocess_text_for_query_expansion(doc)
        tokens.extend(cleaned.split())

    total = len(tokens)
    if total == 0:
        return {}

    freqs = Counter(tokens)
    return {term: freq / total for term, freq in freqs.items()}


def query_expansion(original_query, top_docs, top_n=5, prob_threshold=0.01):
    if not top_docs:
        return original_query

    old_terms = set(preprocess_text_for_query_expansion(original_query).split())
    term_probs = compute_term_distribution(top_docs)

    filtered = [
        (term, prob) for term, prob in term_probs.items()
        if term not in old_terms and prob > prob_threshold
    ]

    top_terms = sorted(filtered, key=lambda x: x[1], reverse=True)[:top_n]

    if not top_terms:
        return original_query

    expansion = " ".join(t for t, _ in top_terms)
    expanded_query = original_query + " " + expansion

    print(f"Query expanded with: {[t for t, _ in top_terms]}")
    return expanded_query


In [63]:
def load_jsonl(file_path):
    titles, texts = [], []
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            obj = json.loads(line)
            text = obj.get("text", "").strip()
            if not text:
                continue

            title = obj.get("title", "").strip()
            if not title or title.lower() == "untitled":
                preview = " ".join(text.split()[:8]) + "..."
                title = f"[Preview] {preview}"

            titles.append(title)
            texts.append(text)
    return titles, texts

In [64]:
def re_retrieve_and_answer(summary, query, index, embedding_model, corpus_chunks, summarizer_fn, titles=None, top_k=5):
    print("\n=== Re-Retrieval Phase ===")

    reformulated_query = summary
    query_embedding = embedding_model.encode(reformulated_query, convert_to_tensor=True)
    D, I = index.search(query_embedding.cpu().numpy().reshape(1, -1), top_k)
    top_docs_reretrieved_indices = I[0].tolist()

    retrieved_docs = []
    top_chunks = []

    for idx in top_docs_reretrieved_indices:
        if idx < len(corpus_chunks):
            top_chunks.append(corpus_chunks[idx])
            retrieved_docs.append({
                "title": titles[idx] if titles else f"Chunk {idx}",
                "text": corpus_chunks[idx],
                "similarity": float(D[0][top_docs_reretrieved_indices.index(idx)])
            })

    print("\n=== Re-Retrieved Documents with Summaries ===")
    per_chunk_summaries = []
    for i, doc in enumerate(retrieved_docs):
        print(f"\nDocument {i+1}:")
        print(f"Title: {doc['title']}")
        print(f"Score: {doc['similarity']:.4f}")
        summary = summarizer_fn(doc['text'])
        per_chunk_summaries.append(summary)
        print(summary)

    combined_summary_input = " ".join(per_chunk_summaries)
    final_summary = summarizer_fn(combined_summary_input)


    return final_summary.strip() if final_summary.strip() else "No relevant summary could be generated."


# Main function

In [65]:
from collections import OrderedDict

if __name__ == "__main__":

    model = SentenceTransformer("multi-qa-mpnet-base-dot-v1", device='cuda')

    file_path = "chunked_documents.jsonl"
    titles, texts = load_jsonl(file_path)
    embeddings = embed_texts(texts, model)
    index = build_index(embeddings)

    while True:
        query = input("\nAsk your travel question (or type 'quit' to exit): ")
        if query.lower() in ['quit', 'exit']:
            print("Exiting.")
            break

        total_start = time.time()

        # Step 1: Retrieval
        retrieval_start = time.time()
        initial_indices, scores = retrieval_top_k(query, model, index, texts, k=5)

        top_docs = []
        retrieved_docs = []

        for pos, i in enumerate(initial_indices[:5]):

            if i < len(titles) and i < len(texts):
                top_docs.append(texts[i])
                retrieved_docs.append({
                    "title": titles[i],
                    "text": texts[i],
                    "similarity": float(scores[pos])
            })
        retrieval_time = time.time() - retrieval_start

        if not retrieved_docs:
            print("No relevant documents found.")
            continue

        # Step 2: Query Expansion
        query_expansion_start = time.time()
        print("\nExpanded Query:")
        expanded_query = query_expansion(query, top_docs)
        print(expanded_query)
        query_expansion_time = time.time() - query_expansion_start

        # Step 3: Summarization
        print("\n Retrieving Documents with Summaries.............")
        summarization_start = time.time()
        chunk_summaries = [summarize_text(doc) for doc in top_docs]
        chunk_summaries = list(OrderedDict.fromkeys(chunk_summaries))  # Deduplicate


        for i, doc in enumerate(retrieved_docs):
            print(f"Document {i+1}:")
            print(f"Title: {doc['title']}")
            print(f"Score: {doc['similarity']:.4f}")
            print(chunk_summaries[i])
            print("\n")

        combined_summary_text = " ".join(chunk_summaries)
        if len(combined_summary_text.split()) > 100:
            refined_summary = summarize_text(combined_summary_text)
        else:
            refined_summary = combined_summary_text
        summarization_time = time.time() - summarization_start

        reretrieve_start = time.time()
        final_answer = re_retrieve_and_answer(
            summary=refined_summary,
            query=query,
            index=index,
            embedding_model=model,
            corpus_chunks=texts,
            summarizer_fn=summarize_text,
            titles= titles
        )
        reretrieve_time = time.time() - reretrieve_start

        print("\n=== Final Answer After Re-retrieval and Summarization ===")
        print(final_answer)

        total_time = time.time() - total_start

        # Latency report
        print("\n=== Latency Report ===")
        print(f"Retrieval time:        {retrieval_time:.2f} sec")
        print(f"Query expansion time:  {query_expansion_time:.2f} sec")
        print(f"Summarization time:    {summarization_time:.2f} sec")
        print(f"Re-retrieval time:     {reretrieve_time:.2f} sec")
        print(f"Total time:            {total_time:.2f} sec")


Batches:   0%|          | 0/75 [00:00<?, ?it/s]


Ask your travel question (or type 'quit' to exit): provide me information about albania

Expanded Query:
Query expanded with: ['–']
provide me information about albania –

 Retrieving Documents with Summaries.............
Document 1:
Title: Albania Travel Guide
Score: 0.6911
Albania has a rich history dating back to the ancient Illyrians and Greeks. Hikers and nature lovers can partake of all the hiking and trekking here. Berat is a UNESCO World Heritage Site and one of the highlights of visiting Albania. Albania's capital is rapidly transforming into a vibrant, cosmopolitan city.


Document 2:
Title: Albania Travel Guide
Score: 0.6532
You won't find many hostels outside of the main tourist cities, but private guest houses are pretty cheap in the countryside anyway. A backpacking budget covers a hostel dorm, cooking your meals, limiting your drinking, using public transportation to get around, and sticking to free and cheap activities like hiking and free tours. On a mid-range budget,